# 00 — Data Pipeline

Loads all raw CSVs from `data/raw/`, validates and cleans them, joins with
`references/level_specs/parameters.json` for IV labels, and writes
analysis-ready parquet files to `data/processed/`.

**Run this notebook first before any analysis notebook.**


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT      = Path('..').resolve()
RAW_DIR   = ROOT / 'data' / 'raw'
PROC_DIR  = ROOT / 'data' / 'processed'
PARAM_PATH = ROOT / 'references' / 'level_specs' / 'parameters.json'

PROC_DIR.mkdir(parents=True, exist_ok=True)
print('Paths OK')
print(f'  raw   : {RAW_DIR}')
print(f'  processed: {PROC_DIR}')


## 1. Load raw CSVs


In [ ]:
csv_files = sorted(RAW_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(f'  {f.name}')


In [ ]:
SCHEMA = [
    'session_id', 'participant_id', 'observer_id', 'level_id',
    'phenomenon', 'condition', 'date', 'wall_time',
    'elapsed_s', 'event_code', 'event_label', 'note'
]

frames = []
for f in csv_files:
    df = pd.read_csv(f, dtype=str)
    missing = [c for c in SCHEMA if c not in df.columns]
    if missing:
        print(f'WARNING {f.name}: missing columns {missing} — skipping')
        continue
    frames.append(df[SCHEMA])

if not frames:
    print('No valid CSV files found. Run a session first, then re-run this notebook.')
else:
    raw = pd.concat(frames, ignore_index=True)
    raw['elapsed_s'] = pd.to_numeric(raw['elapsed_s'], errors='coerce')
    raw['datetime']  = pd.to_datetime(raw['date'] + ' ' + raw['wall_time'],
                                       format='%Y-%m-%d %H:%M:%S.%f', errors='coerce')
    print(f'Loaded {len(raw):,} events from {len(frames)} files')
    print(raw.dtypes)


## 2. Validate


In [ ]:
if 'raw' in dir():
    print('=== Session summary ===')
    summary = (
        raw.groupby(['session_id', 'participant_id', 'phenomenon', 'condition'])
           .agg(n_events=('event_code', 'count'),
                duration_s=('elapsed_s', 'max'))
           .reset_index()
    )
    display(summary)

    print('\n=== Event codes by phenomenon ===')
    display(raw.groupby(['phenomenon', 'event_code']).size()
               .rename('count').reset_index())

    # Flag sessions with no SESSION_END (incomplete)
    ended = raw[raw['event_code'] == 'SESSION_END']['session_id'].unique()
    all_sess = raw['session_id'].unique()
    incomplete = [s for s in all_sess if s not in ended]
    if incomplete:
        print(f'\nWARNING: {len(incomplete)} sessions lack SESSION_END: {incomplete}')
    else:
        print('\nAll sessions have SESSION_END ✓')


## 3. Clean

- Remove SESSION_PAUSE / SESSION_RESUME / SESSION_END meta-events from behavioural analyses
- Correct elapsed_s to exclude paused time


In [ ]:
if 'raw' in dir():
    META_CODES = {'SESSION_PAUSE', 'SESSION_RESUME', 'SESSION_END'}

    # Identify paused intervals per session and subtract from elapsed_s
    def fix_elapsed(grp):
        grp = grp.sort_values('elapsed_s').copy()
        pause_correction = 0.0
        last_pause = None
        for _, row in grp.iterrows():
            if row['event_code'] == 'SESSION_PAUSE':
                last_pause = row['elapsed_s']
            elif row['event_code'] == 'SESSION_RESUME' and last_pause is not None:
                pause_correction += row['elapsed_s'] - last_pause
                last_pause = None
        # elapsed_s from the companion app already excludes paused time,
        # so no adjustment needed — but flag if inconsistency found
        return grp

    raw = raw.groupby('session_id', group_keys=False).apply(fix_elapsed)

    # Behavioural events only
    beh = raw[~raw['event_code'].isin(META_CODES)].copy()
    print(f'Behavioural events: {len(beh):,} (dropped {len(raw) - len(beh):,} meta-events)')


## 4. Join with level parameters


In [ ]:
if 'beh' in dir():
    with open(PARAM_PATH) as fh:
        params = json.load(fh)

    # Build a flat lookup: phenomenon -> spec_file, and other top-level params
    param_rows = []
    for phenom, cfg in params.items():
        if phenom.startswith('_'):
            continue
        param_rows.append({
            'phenomenon': phenom,
            'spec_file':  cfg.get('spec_file', ''),
            'reinforcer': cfg.get('reinforcer', ''),
        })
    param_df = pd.DataFrame(param_rows)

    beh = beh.merge(param_df, on='phenomenon', how='left')
    print('Joined parameters:')
    display(beh[['phenomenon', 'spec_file', 'reinforcer']].drop_duplicates())


## 5. Save processed datasets


In [ ]:
if 'beh' in dir():
    # Master file
    out_all = PROC_DIR / 'all_sessions.parquet'
    beh.to_parquet(out_all, index=False)
    print(f'Saved: {out_all}')

    # One file per phenomenon
    for phenom, grp in beh.groupby('phenomenon'):
        out = PROC_DIR / f'{phenom}.parquet'
        grp.to_parquet(out, index=False)
        print(f'  {phenom}: {len(grp):,} events → {out.name}')

    print('\nPipeline complete.')
